# skin_feature_rebuild 추가 셀: EPI 대조 · 메타 변수 · 그룹 분할

**사용법:** `skin_feature_rebuild.ipynb`의 마지막 셀 뒤에 이 셀들을 복사해 붙이거나, 같은 커널에서 이어서 실행하세요. `final`, `desc`, `cache`, `filtered`, `DESC_COLS`가 메모리에 있어야 합니다.

0. 함수 정의 (한 번만 실행)
1. EPI 대조: 템플릿 생성 → EPI 값 입력 → 비교
2. 메타 변수 추가
3. 그룹 분할 CV (desc / desc+meta / meta-only)

## 0. 함수 정의

In [ ]:
"""
skin_addons.py  --  skin_feature_rebuild.ipynb 뒤에 붙이는 추가 모듈

1) EPI Suite 대조: 샘플 추출 -> 입력용 CSV 템플릿 -> 채운 뒤 비교/플롯
2) unknown_pct / undeclared_pct 를 모델 입력 변수로 (결측·클리핑 처리 포함)
3) group 기준(StratifiedGroupKFold) 학습/검증 분리 + 누수 점검 + feature set 비교

노트북에서:  import skin_addons as SA
"""
import numpy as np
import pandas as pd
from rdkit import Chem

META_COLS = ["unknown_pct", "undeclared_pct"]


# ======================================================================
# 1. EPI Suite 대조
# ======================================================================
def largest_fragment_smiles(smiles):
    """노트북 3단계와 동일한 염 제거 (최대 heavy atom 조각). (smiles, 염제거여부)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, False
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) == 1:
        return Chem.MolToSmiles(mol), False
    big = max(frags, key=lambda m: m.GetNumHeavyAtoms())
    return Chem.MolToSmiles(big), True


def make_epi_template(cache, desc, n=30, seed=0, out_path="epi_check_template.csv"):
    """
    LogP 5분위 층화로 n종 추출 -> EPI Suite에 넣을 SMILES와 빈 칸이 있는 CSV 저장.
    cache: casrn_smiles_cache (casrn, smiles 컬럼)   desc: casrn 인덱스 descriptor 표
    """
    d = desc.reset_index().merge(cache[["casrn", "smiles"]], on="casrn", how="left")
    d = d.dropna(subset=["smiles"]).copy()
    d["logp_bin"] = pd.qcut(d["LogP"], 5, labels=False, duplicates="drop")
    per_bin = int(np.ceil(n / d["logp_bin"].nunique()))
    pick = (d.groupby("logp_bin", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), per_bin), random_state=seed))
              .head(n))

    rows = []
    for _, r in pick.iterrows():
        smi, stripped = largest_fragment_smiles(r["smiles"])
        rows.append({
            "casrn": r["casrn"],
            "smiles_for_EPI": smi,                 # EPI Suite 에는 이 SMILES 를 입력
            "salt_stripped": stripped,
            "MolWt": round(r["MolWt"], 2),
            "rdkit_LogP": round(r["LogP"], 3),
            "rdkit_ESOL_LogS": round(r["ESOL_LogS"], 3),
            # ---- 아래는 직접 채우기 (모르면 빈칸 유지) ----
            "epi_exp_logKow": np.nan,      # EPI의 실측값(Exp Log Kow DB)이 있으면 우선 사용
            "epi_KOWWIN_logKow": np.nan,   # KOWWIN 추정값
            "epi_exp_WS_mgL": np.nan,      # 실측 수용해도 (mg/L)
            "epi_WSKOWWIN_WS_mgL": np.nan, # WSKOWWIN 추정 수용해도 (mg/L)
            "note": "",
        })
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{len(out)}종 템플릿 저장: {out_path}  (염 제거 적용 {out['salt_stripped'].sum()}종)")
    return out


def _metrics(x, y):
    """x=RDKit, y=기준값"""
    from scipy.stats import pearsonr, spearmanr
    diff = x - y
    return {
        "n": len(x),
        "pearson_r": pearsonr(x, y)[0] if len(x) > 2 else np.nan,
        "spearman_rho": spearmanr(x, y)[0] if len(x) > 2 else np.nan,
        "MAE": np.abs(diff).mean(),
        "RMSE": np.sqrt((diff ** 2).mean()),
        "bias(RDKit-ref)": diff.mean(),
    }


def compare_with_epi(filled_path="epi_check_template.csv", outlier_cut=1.0, show=True):
    """
    채워 넣은 CSV 를 읽어 RDKit vs EPI 비교.
    - LogKow : 기준 = 실측(epi_exp_logKow) 우선, 없으면 KOWWIN 추정
    - LogS   : EPI 의 mg/L -> log10(mol/L) 변환 후 RDKit ESOL 과 비교
    반환: (지표 표, 이상치 표)
    """
    import matplotlib.pyplot as plt
    df = pd.read_csv(filled_path)

    df["ref_logKow"] = df["epi_exp_logKow"].fillna(df["epi_KOWWIN_logKow"])
    df["ref_logKow_src"] = np.where(df["epi_exp_logKow"].notna(), "exp",
                                    np.where(df["epi_KOWWIN_logKow"].notna(), "est", ""))
    ws = df["epi_exp_WS_mgL"].fillna(df["epi_WSKOWWIN_WS_mgL"])
    df["ref_logS"] = np.log10(ws / 1000.0 / df["MolWt"])      # mg/L -> mol/L
    df["ref_logS_src"] = np.where(df["epi_exp_WS_mgL"].notna(), "exp",
                                  np.where(df["epi_WSKOWWIN_WS_mgL"].notna(), "est", ""))

    specs = [("LogP vs logKow", "rdkit_LogP", "ref_logKow", "ref_logKow_src"),
             ("ESOL_LogS vs logS", "rdkit_ESOL_LogS", "ref_logS", "ref_logS_src")]
    res, outl = [], []
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, (title, xc, yc, sc) in zip(axes, specs):
        sub = df.dropna(subset=[xc, yc])
        sub = sub[np.isfinite(sub[yc])]
        if len(sub) < 5:
            print(f"[{title}] 유효 쌍 {len(sub)}개 -> 5개 미만이라 지표 생략 (EPI 값을 더 채우세요)")
            ax.set_title(f"{title}  (n={len(sub)}, 부족)")
            continue
        m = _metrics(sub[xc].values, sub[yc].values)
        m["비교"] = title
        m["기준 중 실측 비율"] = (sub[sc] == "exp").mean()
        res.append(m)

        lo = min(sub[xc].min(), sub[yc].min()) - 0.5
        hi = max(sub[xc].max(), sub[yc].max()) + 0.5
        ax.plot([lo, hi], [lo, hi], color="gray", lw=1)
        ax.fill_between([lo, hi], [lo - outlier_cut, hi - outlier_cut],
                        [lo + outlier_cut, hi + outlier_cut], color="gray", alpha=0.12)
        for src, mk in (("exp", "o"), ("est", "s")):
            s2 = sub[sub[sc] == src]
            if len(s2):
                ax.scatter(s2[yc], s2[xc], marker=mk, s=36, label=f"기준={src}")
        bad = sub[(sub[xc] - sub[yc]).abs() > outlier_cut]
        for _, r in bad.iterrows():
            ax.annotate(r["casrn"], (r[yc], r[xc]), fontsize=7, xytext=(3, 3),
                        textcoords="offset points")
        outl.append(bad.assign(비교=title, diff=bad[xc] - bad[yc])
                       [["비교", "casrn", xc, yc, "diff", sc, "salt_stripped"]]
                       .rename(columns={xc: "rdkit", yc: "ref", sc: "ref_src"}))
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel("EPI Suite 기준값"); ax.set_ylabel("RDKit 추정값")
        ax.set_title(f"{title}  (n={len(sub)}, r={m['pearson_r']:.2f}, "
                     f"MAE={m['MAE']:.2f})", fontsize=10)
        ax.legend(fontsize=8)
    plt.tight_layout()
    if show:
        plt.show()

    res_df = pd.DataFrame(res)
    out_df = pd.concat(outl, ignore_index=True) if outl else pd.DataFrame()
    return res_df, out_df


# ======================================================================
# 2. unknown_pct / undeclared_pct 를 입력 변수로
# ======================================================================
def add_meta_features(final):
    """
    - unknown_pct 가 NaN 이면 0 으로 보고 undeclared_pct 를 다시 계산
      (기존 식은 NaN 이 전파돼 undeclared_pct 도 NaN 이 됨)
    - 100 을 넘는 합(원본 % 오류)이 clip(lower=0) 으로 가려지므로 개수를 보고
    - 결측 표시 컬럼(*_missing)을 추가
    반환: (df, 점검 dict)
    """
    df = final.copy()
    report = {"unknown_pct NaN": int(df["unknown_pct"].isna().sum()),
              "undeclared_pct NaN(기존)": int(df["undeclared_pct"].isna().sum())}

    df["unknown_pct_missing"] = df["unknown_pct"].isna().astype(int)
    unk = df["unknown_pct"].fillna(0)
    raw = 100 - df["resolved_pct"] - df["unresolved_structure_pct"] - unk
    report["합계>100 (clip 된 제형)"] = int((raw < 0).sum())
    df["unknown_pct"] = unk
    df["undeclared_pct"] = raw.clip(lower=0)
    return df, report


# ======================================================================
# 3. 그룹 기준 분할 CV
# ======================================================================
def get_formulation_groups(final, group_col="comp_group_id", filtered=None):
    """
    final 에 group_col 이 있으면 그대로 사용.
    없으면 filtered(record 단위, group_id 컬럼)에서 formulation_id 별 그룹을 만든다.
    """
    if group_col in final.columns:
        g = final[group_col]
    else:
        assert filtered is not None and "group_id" in filtered.columns, \
            "group_col 이 final 에 없으면 filtered(group_id 포함)를 넘겨야 합니다"
        per = filtered.groupby("formulation_id")["group_id"].nunique()
        assert (per <= 1).all(), "한 제형에 group_id 가 여러 개입니다 -> 원본 확인 필요"
        m = filtered.groupby("formulation_id")["group_id"].first()
        g = final["formulation_id"].map(m)
    assert g.notna().all(), f"그룹 결측 {int(g.isna().sum())}건 (결측 그룹은 각자 개별 그룹으로 처리하세요)"
    return g.astype(str)


def grouped_cv(df, feature_cols, groups, label="binary_label",
               n_splits=5, seeds=range(5), model_factory=None):
    """
    반복 StratifiedGroupKFold. 각 seed 마다 OOF 확률로 ROC-AUC / PR-AUC 계산.
    - descriptor NaN 행은 제외하고 개수를 반환
    - 매 fold 에서 train/test 그룹 교집합이 0 임을 assert (누수 점검)
    반환: (seed별 지표 표, 정보 dict)
    """
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, average_precision_score

    if model_factory is None:
        model_factory = lambda s: RandomForestClassifier(
            n_estimators=500, class_weight="balanced_subsample",
            min_samples_leaf=2, n_jobs=-1, random_state=s)

    ok = df[feature_cols].notna().all(axis=1).values
    X = df.loc[ok, feature_cols].values
    y = df.loc[ok, label].values
    g = np.asarray(groups)[ok]
    info = {"사용 행": int(ok.sum()), "제외 행(NaN)": int((~ok).sum()),
            "양성": int(y.sum()), "양성 비율(=PR-AUC 기준선)": float(y.mean()),
            "그룹 수": int(len(np.unique(g)))}

    rows, fold_pos = [], []
    for s in seeds:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=s)
        oof = np.zeros(len(y))
        for k, (tr, te) in enumerate(cv.split(X, y, g)):
            assert set(g[tr]).isdisjoint(g[te]), "그룹 누수 발생"
            m = model_factory(s).fit(X[tr], y[tr])
            oof[te] = m.predict_proba(X[te])[:, 1]
            fold_pos.append(int(y[te].sum()))
        rows.append({"seed": s,
                     "roc_auc": roc_auc_score(y, oof),
                     "pr_auc": average_precision_score(y, oof)})
    info["fold당 양성 수(min~max)"] = f"{min(fold_pos)}~{max(fold_pos)}"
    return pd.DataFrame(rows), info


def compare_feature_sets(final, desc_cols, group_col="comp_group_id", filtered=None,
                         drop_esol=True, n_splits=5, seeds=range(5)):
    """descriptor 만 vs descriptor + (unknown_pct, undeclared_pct) 를 같은 분할로 비교."""
    df, rep = add_meta_features(final)
    print("메타 점검:", rep)
    groups = get_formulation_groups(df, group_col, filtered)

    base = [c for c in desc_cols if not (drop_esol and c == "ESOL_LogS")]
    sets = {"desc": base,
            "desc + meta": base + META_COLS,
            "meta only": META_COLS}          # meta 만으로 어느 정도 맞히면 '인공물' 의심
    out, per_seed = [], {}
    for name, cols in sets.items():
        r, info = grouped_cv(df, cols, groups, n_splits=n_splits, seeds=seeds)
        per_seed[name] = r
        out.append({"feature set": name, "n_feat": len(cols),
                    "ROC-AUC": f"{r.roc_auc.mean():.3f} ± {r.roc_auc.std():.3f}",
                    "PR-AUC": f"{r.pr_auc.mean():.3f} ± {r.pr_auc.std():.3f}"})
    print("데이터:", info)
    delta = per_seed["desc + meta"][["roc_auc", "pr_auc"]] - per_seed["desc"][["roc_auc", "pr_auc"]]
    print(f"meta 추가 효과(seed별 평균 차이)  ROC-AUC {delta.roc_auc.mean():+.3f}, "
          f"PR-AUC {delta.pr_auc.mean():+.3f}  "
          f"(개선 seed {int((delta.roc_auc > 0).sum())}/{len(delta)})")
    return pd.DataFrame(out), per_seed


## 1. EPI Suite 대조 (샘플 추출 → 템플릿)

`epi_check_template.csv`가 생성됩니다. EPI Suite에 `smiles_for_EPI`를 입력해 `epi_exp_logKow`, `epi_KOWWIN_logKow`, `epi_exp_WS_mgL`, `epi_WSKOWWIN_WS_mgL` 열을 채우고 저장하세요.

In [ ]:
tmpl = make_epi_template(cache, desc, n=30, seed=0, out_path='epi_check_template.csv')
tmpl.head()

### EPI 값 입력 후 비교 플롯

In [ ]:
res, outliers = compare_with_epi('epi_check_template.csv', outlier_cut=1.0)
res

In [ ]:
outliers

## 2. 메타 변수 추가 (`unknown_pct`, `undeclared_pct`)

In [ ]:
final = add_meta_features(final)
final[META_COLS + ['unknown_pct_missing']].describe()

## 3. 그룹 기준 학습/검증 분할

`final`에 `comp_group_id`가 있으면 그대로 쓰고, 없으면 `filtered['group_id']`에서 제형별로 가져옵니다. 양성이 29개뿐이라 fold당 5~6개이므로 시드별 편차를 함께 보세요.

In [ ]:
tab, per_seed = compare_feature_sets(
    final, DESC_COLS,
    group_col='comp_group_id',
    filtered=filtered,
    drop_esol=True,
    n_splits=5, seeds=range(5),
)
tab

**읽는 법:** meta-only AUC가 높으면 데이터셋 artifact 신호이며 화장품(INCI)으로는 옮겨지지 않을 수 있습니다. `comp_group_id`는 조성이 동일한 쌍둥이만 묶으므로 Tanimoto ≥0.9 근접 쌍둥이 누수는 남을 수 있습니다.